# Can an acquisition branch on the frozen PFN reproduce every arm's ranking? Per-arm distillation

## Description

<!-- Written 2026-11-13 10:15 before the first run (pre-registration). Idea: research_log/decisions/0009 (Tim, 2026-11-13). Issue #13. -->

**Question.** Decision 0009 proposes an acquisition branch beside the frozen PFN: acquisition tokens at the observed locations read the PFN's train-context activations layer by layer, talk to each other, and are queried at candidate points. Before any policy is trained on it: **can this branch, reading only the train-context activations, reproduce the ranking of every arm (EI, PI, UCB, MES, mean) over candidates, well enough that maximizing its output is the same policy as the arm?** If yes, the backbone holds what all reference policies need, and policy improvement can start from them.

**Setup** (`experimental/acq_branch.py`, `tasks/acq_distill.py`, `analysis/acq_distill.py`). Frozen `pfn_large` (8 layers, width 256), fp32, noise-free prior tasks. One sweep member per input dimension: d = 1 (n_init 3, T = 15), d = 2 (4, 20), d = 4 (8, 40); contexts stay ≤ 48 points.
- *Branch.* 8 blocks, width 192, 4 heads; block b reads PFN layer b. Acquisition tokens start from the location only; a budget token (share used, share remaining, context size) joins their self-attention; query tokens start from the location and the budget. 5 output columns, one per arm. The PFN gets no gradient.
- *Training states.* 1500 prior tasks (split 4, offset 60000). Per task one full BO run per arm (deterministic arms, candidate seed 2) and 2 runs of uniformly random points; a state is a prefix of a run.
- *Teacher.* Each arm's acquisition value at the state's seed-0 candidate set (1024 Sobol points + 192 perturbations of the incumbent, the set the arms use in notebooks 12–15) and at 256 fresh queries (uniform, and perturbations of the incumbent and of observed points at scales 1e-3 to 0.3). MES's f\* samples come from the seed-0 set, so every query's value depends on the data and x only.
- *Loss.* Per arm, within the state: cross-entropy towards the arm's best candidate, plus a pairwise logistic loss on random pairs and on (arm's top 8, any) pairs. Only the order matters, so the arms' different scales play no role. 8000 steps of 48 states.
- *Evaluation* on 300 held-out tasks (offset 70000): runs of every arm under seed 0 and one random run per task, at every step.
  - `cand0`: the deployment candidate set (seed 0). `cand1`: a candidate set with unseen Sobol points (seed 1). `dense`: 1024 grid points (d ≤ 2) or Sobol points.
  - Per state and arm: does the branch pick the arm's best candidate (`top1`); what share of candidates the arm ranks above the branch's pick (`rank_pct`; a pick is *good* if ≤ 1%); Spearman over all candidates and over the arm's top quarter.
  - *Closed loop:* BO with the branch's column as the acquisition, against the arm itself, on the same tasks and candidates. Terminal log10 regret (ε = 1e-6), paired; share of runs that are identical step by step.
  - *Pictures* (d ≤ 2): the arm's score and the branch's score over the dense set for example states, to see artifacts.

**What to look for.**
1. d = 1 first: do the learned scores follow the arms over the whole domain, including between and at the observed points and at the incumbent?
2. Per arm and d: share of good picks on `cand0`, `cand1`, `dense`; does it hold on unseen query points?
3. Closed loop: is the regret of the distilled arm the same as the arm's?
4. Where it fails: by context size, by arm (PI and EI have large flat regions; MES depends on the candidate set through f\*).

**GO / NO-GO** (rule `analysis/acq_distill.py::distill_verdict`, per member).
- **GO**: for every arm, the branch's pick on `cand0` is good in ≥ 90% of states, and its closed-loop log10 regret is within 0.1 of the arm's. Next: the improvement step on this backbone (decision 0009 §2).
- **NO-GO**: for some arm, good picks in < 50% of states. Then the train-context activations as read here do not carry the arm; next: the `query_state` ablation (the branch also sees the PFN's query state) to tell an architecture problem from a training problem.
- **INCONCLUSIVE**: otherwise.

**Limits.**
- Distillation only: this shows that the branch can represent and learn the reference policies, not that it can improve on them.
- Agreement in ranking, not in value: the branch's scores are on an arbitrary scale.
- Ties: where an arm's values are tied over many candidates (PI or EI of 0), any of them counts as its best.
- Small picks differences can still change a run, since noise-free runs amplify them; hence the closed-loop check.
- pfn_large within its valid range (n ≤ 100, d ≤ 6).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anytime_acq.analysis import load_snapshot, show_header
from anytime_acq.analysis.acq_distill import distill_verdict

SNAPSHOT = "results/acq_distill/<sweep_id>"  # set after `just snapshot`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

In [ ]:
runs = snap.runs
print("member status:", runs.status.value_counts().to_dict())
dcol = [c for c in runs.columns if c.endswith("task.d")][0]
display(runs[[dcol, "status", "branch_params", "time/train_s", "time/evaluate_s"]].round(0))
def tab(name):
    t = snap.table(name)
    return t.merge(runs[["member", dcol]].rename(columns={dcol: "d"}), on="member") if "member" in t else t.assign(d=runs[dcol].iloc[0])
train, states, closed, curves, summ = (tab(k) for k in ("train", "states", "closed", "curves", "summary"))
ARMS = list(summ.arm.unique()); DS = sorted(states.d.unique())

## 1. Training

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2), dpi=90)
for d, g in train.groupby("d"):
    for a, k in zip(ax, ("loss", "good_pick", "spearman")):
        a.plot(g.step, g[k], label=f"d = {d}"); a.set_title(k + " (training batches)"); a.set_xlabel("step"); a.grid(alpha=.3)
ax[0].set_yscale("log"); ax[0].legend(); plt.show()

## 2. Pictures: the arms and the learned scores over the whole domain (d = 1), and as maps (d = 2)

In [ ]:
def rank01(v, valid):
    r = pd.Series(np.where(valid, v, np.nan)).rank(pct=True).to_numpy()
    return r
if 1 in DS:
    m = int(runs.loc[runs[dcol] == 1, "member"].iloc[0])
    ex, ctx = snap.table("examples"), snap.table("examples_ctx")
    ex, ctx = (t[t.member == m] if "member" in t else t for t in (ex, ctx))
    ids = [e for e in ex.example.unique() if e.startswith(("task0_", "task1_", "task2_"))]
    for eid in ids:
        g, c = ex[ex.example == eid].sort_values("x0"), ctx[ctx.example == eid]
        fig, ax = plt.subplots(1, len(ARMS) + 1, figsize=(3.4 * (len(ARMS) + 1), 2.8), dpi=80)
        ax[0].plot(g.x0, g.f, "k", lw=1, label="f"); ax[0].plot(g.x0, g.pfn_mean, "C0", lw=1, label="PFN mean"); ax[0].plot(c.x0, c.y, "ro", ms=4)
        ax[0].set_title(f"{eid} (n = {len(c)})", fontsize=9); ax[0].legend(fontsize=7)
        for a, arm in zip(ax[1:], ARMS):
            t, s, v = g[f"teacher:{arm}"].to_numpy(), g[f"student:{arm}"].to_numpy(), g.valid.to_numpy()
            a.plot(g.x0, rank01(t, v), "k", lw=1.2, label="arm (rank)"); a.plot(g.x0, rank01(s, v), "C1", lw=1, alpha=.9, label="branch (rank)")
            a.axvline(g.x0.to_numpy()[np.where(v, t, -np.inf).argmax()], color="k", ls=":", lw=1); a.axvline(g.x0.to_numpy()[np.where(v, s, -np.inf).argmax()], color="C1", ls="--", lw=1)
            for x in c.x0: a.axvline(x, color="r", lw=.3, alpha=.5)
            a.set_title(arm, fontsize=9)
        ax[1].legend(fontsize=7); plt.tight_layout(); plt.show()
    print("black: the arm's value as a rank over the grid (1 = best), dotted line = its argmax; orange: the branch, dashed = its argmax; red: observed points")

In [ ]:
if 2 in DS:
    m = int(runs.loc[runs[dcol] == 2, "member"].iloc[0])
    ex, ctx = snap.table("examples"), snap.table("examples_ctx")
    ex, ctx = (t[t.member == m] if "member" in t else t for t in (ex, ctx))
    for eid in [e for e in ex.example.unique() if e.startswith(("task0_ei", "task1_ei", "task0_random"))][:6]:
        g, c = ex[ex.example == eid], ctx[ctx.example == eid]
        k = int(round(np.sqrt(len(g))))
        fig, ax = plt.subplots(2, len(ARMS), figsize=(2.6 * len(ARMS), 5.2), dpi=80)
        for j, arm in enumerate(ARMS):
            for i, who in enumerate(("teacher", "student")):
                z = rank01(g[f"{who}:{arm}"].to_numpy(), g.valid.to_numpy()).reshape(k, k)
                ax[i, j].imshow(z.T, origin="lower", extent=(0, 1, 0, 1), cmap="viridis"); ax[i, j].plot(c.x0, c.x1, "r.", ms=3)
                ax[i, j].set_title(f"{'arm' if i == 0 else 'branch'}: {arm}", fontsize=8); ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
        fig.suptitle(f"{eid} (n = {len(c)}), rank over the grid", fontsize=9); plt.tight_layout(); plt.show()

## 3. Agreement with the arms: candidate sets and the dense set

In [ ]:
agg = states.groupby(["d", "query_set", "arm"]).agg(top1=("top1", "mean"), good_pick=("rank_pct", lambda x: (x <= 0.01).mean()),
        rank_pct_p90=("rank_pct", lambda x: x.quantile(.9)), spearman=("spearman", "mean"), spearman_top=("spearman_top", "mean"))
print("good_pick: share of states where the arm ranks at most 1% of the candidates above the branch's pick")
display(agg.good_pick.unstack("arm").round(3)); display(agg.top1.unstack("arm").round(3)); display(agg.spearman.unstack("arm").round(3)); display(agg.spearman_top.unstack("arm").round(3))
fig, ax = plt.subplots(1, len(DS), figsize=(5 * len(DS), 3.2), dpi=90, squeeze=False)
for a, d in zip(ax[0], DS):
    g = states[(states.d == d) & (states.query_set == "cand0")].assign(good=lambda x: x.rank_pct <= 0.01).groupby(["step", "arm"]).good.mean().unstack("arm")
    g.plot(ax=a); a.set_title(f"d = {d}: good picks on cand0 by step"); a.set_ylim(0, 1.02); a.grid(alpha=.3)
plt.show()
print("by kind of run the state comes from (cand0):")
display(states[states.query_set == "cand0"].assign(good=lambda x: x.rank_pct <= 0.01).groupby(["d", "run"]).good.mean().unstack("run").round(3))

## 4. Closed loop: the branch's column as the acquisition, against the arm

In [ ]:
display(summ.set_index(["d", "arm"])[["cl:diff", "cl:lo", "cl:hi", "cl:same_run", "cl:same_steps"]].round(3))
print("cl:diff = log10 terminal regret of the branch minus the arm's (positive = branch worse); same_run: share of runs identical step by step")
fig, ax = plt.subplots(1, len(DS), figsize=(5 * len(DS), 3.2), dpi=90, squeeze=False)
for a, d in zip(ax[0], DS):
    for i, arm in enumerate(ARMS):
        for pol, ls in (("teacher", "-"), ("student", "--")):
            g = curves[(curves.d == d) & (curves.arm == arm) & (curves.policy == pol)]
            a.plot(g.step, g.log_regret, ls, color=f"C{i}", label=arm if pol == "teacher" else None)
    a.set_title(f"d = {d}: mean log10 regret (solid: arm, dashed: branch)"); a.set_xlabel("step"); a.grid(alpha=.3); a.legend(fontsize=7)
plt.show()

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.

In [ ]:
for d, g in summ.groupby("d"):
    print(f"d = {d}: pre-registered rule: {distill_verdict(g)}; good picks on cand0 per arm:", dict(zip(g.arm, g["cand0:good_pick"].round(3))))